# AquaPINN Sequim Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/01_Sequim_Quickstart.ipynb)

Run the supplied Sequim example in four steps: set up the notebook, choose inputs and tracking settings, run the existing Sequim launcher, and review the results.


## 1. Set up

Run once. Colab saves results to Google Drive; local Jupyter saves them in `AquaPINN_Colab_Runs`.


In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six tomli-w

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "AquaPINN_Tracker3D").is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/AquaPINN_tmp")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        REPO_URL, str(REPO_DIR)], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    OUTPUT_PARENT = REPO_DIR / "AquaPINN_Colab_Runs"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
print("Ready. Results will be saved in:", OUTPUT_PARENT)


## 2. Choose inputs and tracking settings

Edit this block. Paths can point to files in Google Drive. TagID 2 is the bundled Sequim desktop example tag.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Input files
MAT_FILE = REPO_DIR / "Examples/Sequim/data/DriftingTags_timeCorrected.mat"
PHONE_FILE = REPO_DIR / "Examples/Sequim/config/sequim_testing_node_locs_w_beacons.csv"
TAG_FILE = REPO_DIR / "Examples/Sequim/config/DriftingTags.csv"
TEST_FILE = REPO_DIR / "Examples/Sequim/config/EA_test_description.csv"
GPS_FILE = REPO_DIR / "Examples/Sequim/config/drifting_GPS_day1_local_system.csv"

# Tracking settings
METHOD = "AML"  # AML, NN, or NNplus
NNPLUS_SAMPLES = 5
TIME_START = "2024-2-21 11:30:00"
TIME_END = "2024-2-21 13:30:00"
TIME_ZONE_HOURS = 0
TEMPERATURE_C = 8.0
HORIZONTAL_ONLY = True
DEVICE = "Automatic"  # Automatic or CPU only

display(pd.DataFrame({"Setting": ["Tag ID", "Method", "Time range", "Timezone", "Temperature (C)", "2D only", "Device"],
                        "Value": [2, METHOD, f"{TIME_START} to {TIME_END}", TIME_ZONE_HOURS, TEMPERATURE_C, HORIZONTAL_ONLY, DEVICE]}))


## 3. Run Sequim

This runs the project’s existing `main_Sequim.py` workflow. Sequim detections are already time-corrected, so no new synchronization step is needed.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, sys, uuid
import pandas as pd
import tomllib
import tomli_w

SEQ_RUN = REPO_DIR / "Examples/Sequim/run"
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(SEQ_RUN))

import AquaPINN_Tracker3D as pinn
from AquaPINN_Tracker3D import config
from convertDetections import convert_mat_to_synced_pickle, find_mat_tag_indices

# Create a fresh output folder and keep the selected MAT input with the results.
RUN_ID = uuid.uuid4().hex
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_PARENT / f"Sequim_{stamp}_{RUN_ID[:8]}"
INPUTS = RUN_DIR / "inputs"
RESULTS = RUN_DIR / "results"
INPUTS.mkdir(parents=True)
RESULTS.mkdir()
MAT_STAGED = INPUTS / MAT_FILE.name
shutil.copy2(MAT_FILE, MAT_STAGED)

# Start from the supplied Sequim settings and apply the choices above.
with (SEQ_RUN / "params.toml").open("rb") as stream:
    params = tomllib.load(stream)
params.update({
    "TagID": [2],
    "phoneFile": [str(PHONE_FILE)],
    "tagFile": [str(TAG_FILE)],
    "testFile": [str(TEST_FILE) if TEST_FILE else ""],
    "GPSFile": [str(GPS_FILE) if GPS_FILE else ""],
    "dataFolder": [str(INPUTS)],
    "output_folder": [str(RESULTS)],
    "time_start": [TIME_START],
    "time_end": [TIME_END],
    "timeZone": [TIME_ZONE_HOURS],
    "temperature_ref": [TEMPERATURE_C],
    "ignoreZ": ["True" if HORIZONTAL_ONLY else "False"],
    "trackMethod": ["AML" if METHOD == "AML" else "NN"],
    "nSamples": [1 if METHOD != "NNplus" else NNPLUS_SAMPLES],
})
PARAMS_FILE = RUN_DIR / "params_effective.toml"
PARAMS_FILE.write_text(tomli_w.dumps(params), encoding="utf-8")
(RUN_DIR / "params_effective.json").write_text(json.dumps(params, indent=2), encoding="utf-8")

# Configure AquaPINN and prepare the Sequim receiver system.
config.oneKey_configure(use_cuda=(DEVICE == "Automatic"), precision=64, seed=42)
train_params = {key: value[0] for key, value in params.items()}
tag_id = train_params["TagID"]
tag_code = str(pd.read_csv(TAG_FILE).iloc[tag_id]["Code"]).strip()
pinn.getLogger(logFile=str(RESULTS / f"history_Tag{tag_id}.log"))

# Sequim detections are already time-corrected: copy receiver locations as synced,
# convert the selected MAT detections, then run the tracker.
phone_synced = RESULTS / (PHONE_FILE.stem + "_synced.csv")
shutil.copy2(PHONE_FILE, phone_synced)
loc_sys = pinn.localizationSystem().deploy(str(PARAMS_FILE), doSync=False)
mat_indices = find_mat_tag_indices(str(MAT_STAGED), list(loc_sys.tagInfo["code"]))
if mat_indices != sorted(mat_indices):
    raise ValueError("Tag catalogue order does not match the order in the MAT file.")
synced_pickle = loc_sys.getSyncedDecodesFile(
    train_params["tagFile"], "Tag", newFolder=train_params["output_folder"])
convert_mat_to_synced_pickle(
    mat_file=str(MAT_STAGED), out_pickle=synced_pickle,
    badPhones=train_params["badPhones"], tagIndices=mat_indices)
pinn.tracker.run(str(PARAMS_FILE), loc_sys)

# Save the small run record the desktop Simple Example can reopen.
track_rel = f"results/Track/track/track_Tag{tag_id}_{tag_code}_{METHOD}.csv"
TRACK_CSV = RUN_DIR / track_rel
run_record = {
    "schema": 1, "run_id": RUN_ID, "state": "completed", "method": METHOD,
    "track_csv": track_rel,
    "track_sha256": hashlib.sha256(TRACK_CSV.read_bytes()).hexdigest(),
}
(RUN_DIR / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print("Run complete. Results:", RUN_DIR)


## 4. Review results

The run folder contains the track, plots, parameters, log, and desktop Simple Example run record. Keep the whole folder if you want to reopen the run later.


In [ ]:
import pandas as pd
from IPython.display import display, Image

track = pd.read_csv(TRACK_CSV)
display(track.head())

for figure in sorted(RESULTS.rglob("*.jpg"))[:4]:
    display(Image(filename=str(figure)))

print("Saved run folder:", RUN_DIR)
print("Desktop run record:", RUN_DIR / "run.json")
